# Audyt wyników TrackingAI — 30.09.2026

Cel: niezależnie przeliczyć ranking ze źródłowego CSV i odróżnić średnią od ostatniej próby oraz rekordu. Ten notebook używa wyłącznie biblioteki standardowej Pythona i zapisanych danych; nie pobiera ani nie wykonuje kodu źródłowej strony.

Źródła: [CSV](https://www.trackingai.org/app/database/proj_IQ/score_logs/iq/daily_logs.csv), [obliczenia rankingu](https://www.trackingai.org/assets/js/charts/charts_IQ.js), [konfiguracja aliasów](https://www.trackingai.org/assets/js/config.js). Metadane pobrania i skróty SHA-256 są w `tracking-receipt.json`.

Wyniki dotyczą quizów TrackingAI, nie psychometrycznie potwierdzonego IQ. Aktualne nazwy aliasów mogą obejmować różne wersje modelu. Daty prób nie są datami premier. Źródło nie deklaruje strefy czasowej dat prób.


In [1]:
import csv, hashlib, json, math, re
from collections import defaultdict
from datetime import datetime
from pathlib import Path

candidates = [Path.cwd(), Path.cwd() / 'research/ai', Path.cwd() / 'plotwist/research/ai']
data = next(path for path in candidates if (path / 'tracking-receipt.json').is_file())
receipt = json.loads((data / 'tracking-receipt.json').read_text(encoding='utf-8'))
raw = (data / 'tracking-iq.csv').read_bytes()
digest = hashlib.sha256(raw).hexdigest()
assert digest == receipt['downloads'][0]['sha256']
rows = list(csv.DictReader(raw.decode('utf-8-sig').splitlines()))
assert len(rows) == receipt['sourceRows'] == 5705
print(json.dumps({'checkedAt': receipt['checkedAt'], 'sourceRows': len(rows), 'sha256': digest}, indent=2))


{
  "checkedAt": "2026-09-30T14:57:34.099Z",
  "sourceRows": 5705,
  "sha256": "273594b522f71174ae88c7934a76d3974bc14432f573243a9723f2cbe262169a"
}


## Indywidualne próby

Wzory przepisano z kodu TrackingAI. Mensa używa `test_score`, Offline używa `valid_test_score`. Najpierw zaokrąglamy każdą próbę, dopiero potem średnią. `Overall` nie jest oddzielną próbą IQ i jest pomijane. Parser przyjmuje także miesiąc lub godzinę bez zera na początku.


In [2]:
def round_js(value):
    return math.floor(value + 0.5)

def iq(row):
    if row['test_source'] == 'Mensa Norway':
        return round_js(63.5 + 3 * (float(row['test_score']) - 5.833))
    if row['test_source'] == 'Offline Test':
        return round_js(63.5 + 3 * 0.8823 * ((float(row['valid_test_score']) - 3) * (35 / 14)))
    return None

runs = []
for index, row in enumerate(rows):
    score = iq(row)
    if score is None:
        continue
    date = datetime.strptime(row['date_time'].strip(), '%m/%d/%Y %H:%M:%S').isoformat()
    runs.append({'id': f'tracking-{index}', 'alias': row['ai_name'], 'test': row['test_source'], 'date': date, 'score': score})

recovered = sum(not re.fullmatch(r'\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2}', row['date_time']) for row in rows if iq(row) is not None)
assert len(runs) == receipt['validIndividualRuns'] == 3824
assert recovered == 33
assert max(run['date'] for run in runs) == receipt['latestRunAt']
print(json.dumps({'individualRuns': len(runs), 'recoveredByFlexibleDateParser': recovered, 'latestRunAt': receipt['latestRunAt']}, indent=2))


{
  "individualRuns": 3824,
  "recoveredByFlexibleDateParser": 33,
  "latestRunAt": "2026-09-29T23:39:25"
}


## Średnia rankingu, ostatnia próba i rekord

Średnia używa ostatnich maksymalnie 7 prób danego aliasu i testu, a nie 7 dni. Źródłowa konfiguracja może nadpisać długość okna; w tym pobraniu nie ma nadpisań. Tekst i Vision pozostają osobnymi pozycjami. Wykluczamy aliasy wyłączone z bieżącego rankingu przez źródło. Rekord poniżej to maksimum całej dostępnej historii aliasu i testu — niekoniecznie jednej wersji modelu.


In [3]:
config = receipt['config']
groups = defaultdict(list)
for run in runs:
    if run['alias'] not in config['excludedAliases']:
        groups[(run['test'], run['alias'])].append(run)

verified = []
for test, slug in [('Mensa Norway', 'mensa'), ('Offline Test', 'offline')]:
    dataset = json.loads((data / f'../../public/ai/tracking-{slug}-ranking.json').read_text(encoding='utf-8'))
    generated = {row['sourceAlias']: row for row in dataset['rows']}
    aliases = {alias for source, alias in groups if source == test}
    assert aliases == set(generated)
    for alias in aliases:
        history = sorted(groups[(test, alias)], key=lambda run: run['date'], reverse=True)
        window = config['windowByAlias'].get(alias, config['defaultWindow'])
        chosen = history[:window]
        mean = round_js(sum(run['score'] for run in chosen) / len(chosen))
        row = generated[alias]
        assert (row['score'], row['latestScore'], row['recordScore'], row['sampleCount']) == (mean, history[0]['score'], max(run['score'] for run in history), len(chosen))
        assert [(run['id'], run['date'], run['score']) for run in row['runs']] == [(run['id'], run['date'], run['score']) for run in chosen]
        assert row['releaseDate'] is None and row['modelVersionUnverified'] is True
        verified.append({'test': test, 'model': row['model'], 'mean': mean, 'latest': row['latestScore'], 'record': row['recordScore'], 'n': len(chosen)})

print(f'Verified: {len(verified)} ranking entries with complete run evidence')
print(f"{'Model':<36} {'Mean':>5} {'Latest':>7} {'Record':>7} {'N':>3}")
for row in sorted((row for row in verified if row['test'] == 'Mensa Norway'), key=lambda row: (-row['mean'], row['model'])):
    print(f"{row['model']:<36} {row['mean']:>5} {row['latest']:>7} {row['record']:>7} {row['n']:>3}")
assert len([row for row in verified if row['test'] == 'Mensa Norway']) == 25
assert len([row for row in verified if row['test'] == 'Offline Test']) == 29


Verified: 54 ranking entries with complete run evidence
Model                                 Mean  Latest  Record   N
GPT 6 Astra Ultra (Vision)             151     151     151   1
Claude-5.1 Fable                       147     151     151   2
Claude-5.1 Fable (Vision)              145     145     145   1
GPT 6.1 Sol Ultra                      145     139     151   7
Claude-5.5 Opus                        144     142     148   7
Claude-5.5 Opus MAX                    144     148     148   7
GPT 6 Astra Ultra                      144     142     145   2
Qwen 3.8 Max                           143     142     148   7
GPT 5.6 Terra Ultra                    142     142     142   6
Gemini 3.8 Flash                       142     145     148   7
Grok 4.7 High                          142     142     151   7
GPT 5.6 Terra Ultra (Vision)           141     142     142   4
GPT 6 Luna Max                         140     127     148   5
GPT 6 Luna Max (Vision)                139     133     145   4

## Wnioski

- Najnowszy wpis źródłowy: 29.09.2026, 23:39:25 (strefa źródła nieznana). Sprawdzenie źródła: 30.09.2026, 14:57 UTC. CSV był identyczny z dotychczasową kopią.
- Poprawiony parser przywraca 33 starsze próby pomijane przez wcześniejszy sztywny format daty.
- Średnia, ostatnia próba i rekord dają różne wartości, np. GPT 6.1 Sol Ultra: 145 / 139 / 151. Claude-5.1 Fable ma dwie próby (151 i 142), więc zaokrąglona średnia wynosi 147.
- Bieżący snapshot Mensa reprodukuje wszystkie 25 wartości ze zrzutu użytkownika. Zgodność z samym zrzutem sprawdza także `tests/tracking-iq.test.mjs`.
- Nie należy traktować bieżących nazw aliasów jako udokumentowanej historii premier lub porównania wyłącznie najnowszych wersji modeli.
